# Basin delineation

In [ ]:
# general
import pandas as pd
from glob import glob
import os

# spatial
import geopandas as gpd
import rioxarray as rxr
from geocube.vector import vectorize
import whitebox

from utils.config import find_repo_root, get_data_root, get

path_dem = str(get_data_root() / get('dirs')['dem'])
wd = find_repo_root()
os.chdir(wd)

wbt = whitebox.WhiteboxTools()
wbt.set_compress_rasters(True)
print(wbt.version())

## Data (ARG + PERU)

In [ ]:
# Load data for both regions
regions = {
    "SENAMHI": "data/SENAMHI_PERU",
    "SNHI": "data/SNHI_ARG"
}

for region_name, data_path in regions.items():
    q_data = pd.read_csv(f"{data_path}/{region_name}_daily_1950_2024.csv", index_col=0, parse_dates=["date"])
    q_metadata = pd.read_csv(f"{data_path}/{region_name}_metadata.csv", encoding="latin", index_col=0)
    q_metadata = q_metadata.loc[q_data.columns]
    
    q_shape = gpd.GeoDataFrame(q_metadata, geometry=gpd.points_from_xy(x=q_metadata.gauge_lon, y=q_metadata.gauge_lat), crs=f"EPSG:{get('epsg_wgs84')}")
    q_shape["geometry"].to_file(f"temp/stream_gauges_{region_name.split('_')[0]}.shp")

## Delineation and conversion

In [ ]:
regions = {
    "PERU": "SENAMHI",
     "ARG": "SNHI"
}

for dem, region in regions.items():

    # perform flow accumulation workflow (fill depressions, calculate flow directions and flow accumulation)
    wbt.flow_accumulation_full_workflow(
        dem = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m.tif", 
        out_dem = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_filled.tif", 
        out_pntr = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_flow_dir.tif", 
        out_accum = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_flow_acc.tif"
        );

    # extract streams (threshold needs to be tuned)
    wbt.extract_streams(
        flow_accum = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_flow_acc.tif",
        output = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_streams.tif",
        threshold = 3, # min area, not number of cells
        zero_background = True
        );

    # snap pour points to the nearest stream
    wbt.jenson_snap_pour_points(
        streams = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_streams.tif",
        pour_pts = wd / f"temp/stream_gauges_{region}.shp",
        output = wd / f"temp/stream_gauges_{region}_snap.shp",
        snap_dist = 20
        );

    # delineate basins
    wbt.unnest_basins(
        d8_pntr = path_dem + f"/DEM_{dem}_{get('dem_resolution')}m_flow_dir.tif", 
        pour_pts = wd / f"temp/stream_gauges_{region}_snap.shp", 
        output = wd / f"temp/basins_{region}.tif"
        );
    
    basins = glob(str(wd / f"temp/basins_{region}*.tif"))
    gauge_ids = gpd.read_file(wd / f"temp/stream_gauges_{region}_snap.shp").set_index("index")

    # Convert basins to vector
    gdf = []
    for basin in basins:
        basin = rxr.open_rasterio(basin)
        basin.name = "id"  # Assign a name to avoid warning
        gdf.append(vectorize(basin))

    gdf = gpd.GeoDataFrame(pd.concat(gdf, ignore_index=True), crs=basin.rio.crs)
    gdf = gdf.dissolve(by="id").reset_index()
    gdf["gauge_id"] = gauge_ids.index
    gdf[["gauge_id", "geometry"]].to_file(wd / "data" / f"{region}_{dem}" / f"basins_{region}.gpkg")

In [ ]:
# Remove temporary files
temp_files = glob(path_dem + f"/DEM_*_{get('dem_resolution')}m_*") + glob(str(wd / "temp" / "*"))
for file in temp_files:
    os.remove(file)